# Dev checks: API keys + LLM + Portkey + Gemini embedding
Run the cells top to bottom. Each check prints one PASS/FAIL line.

In [ ]:
import os, sys
# notebook lives in tests/: walk up to the project root (the folder containing app/)
root = os.getcwd()
while not os.path.isdir(os.path.join(root, 'app')) and os.path.dirname(root) != root:
    root = os.path.dirname(root)
os.chdir(root)
sys.path.insert(0, root)
from app.config import settings
def mask(k): return (k[:4] + '...' + k[-3:]) if k else 'MISSING'
print('GROQ_MODEL    :', settings.GROQ_MODEL)
print('GROQ key      :', mask(settings.GROQ_API_KEY))
print('PORTKEY key   :', mask(os.getenv('PORTKEY_API_KEY')))
print('PORTKEY slug  :', os.getenv('PORTKEY_PRIMARY_SLUG'))
print('PORTKEY config:', os.getenv('PORTKEY_PRIMARY_CONFIG_ID') or 'MISSING')

## 1. Provider API keys

In [ ]:
import httpx
TIMEOUT = 15

def check_groq(api_key, model=None):
    r = httpx.get('https://api.groq.com/openai/v1/models', headers={'Authorization': f'Bearer {api_key}'}, timeout=TIMEOUT)
    if r.status_code != 200:
        return False, f'HTTP {r.status_code}: {r.text[:200]}'
    ids = [m['id'] for m in r.json().get('data', [])]
    if model and model not in ids:
        return False, f"key is valid but model '{model}' is not available. Available: {ids}"
    return True, f'{len(ids)} models visible'

def check_gemini(api_key):
    r = httpx.get('https://generativelanguage.googleapis.com/v1beta/models', params={'key': api_key}, timeout=TIMEOUT)
    return r.status_code == 200, f'HTTP {r.status_code}'

def check_qdrant(url, api_key):
    headers = {'api-key': api_key} if api_key else {}
    r = httpx.get(f"{url.rstrip('/')}/collections", headers=headers, timeout=TIMEOUT)
    return r.status_code == 200, f'HTTP {r.status_code}'

def check_anthropic(api_key):
    r = httpx.get('https://api.anthropic.com/v1/models', headers={'x-api-key': api_key, 'anthropic-version': '2023-06-01'}, timeout=TIMEOUT)
    return r.status_code == 200, f'HTTP {r.status_code}'

checks = [
    ('GROQ_API_KEY', settings.GROQ_API_KEY, lambda k: check_groq(k, settings.GROQ_MODEL)),
    ('GROQ_FALLBACK_API_KEY', settings.GROQ_FALLBACK_API_KEY, lambda k: check_groq(k, settings.GROQ_MODEL)),
    ('GEMINI_API_KEY', settings.GEMINI_API_KEY, check_gemini),
    ('QDRANT', settings.QDRANT_URL, lambda u: check_qdrant(u, settings.QDRANT_API_KEY)),
    ('ANTHROPIC_API_KEY', settings.ANTHROPIC_API_KEY, check_anthropic),
]
for name, value, fn in checks:
    if not value:
        print(f'[MISSING] {name}')
        continue
    try:
        ok, msg = fn(value)
    except Exception as e:
        ok, msg = False, str(e)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {msg}")

## 2. Groq LLM direct (bypasses the gateway)

In [ ]:
from langchain_groq import ChatGroq
try:
    llm = ChatGroq(model=settings.GROQ_MODEL, api_key=settings.GROQ_API_KEY, temperature=0)
    print('PASS Groq ->', llm.invoke('In one sentence, what is a Kubernetes Pod?').content)
except Exception as e:
    print('FAIL Groq ->', str(e)[:200])

## 3. Planner routing: greetings must be CONVERSATIONAL (no retrieval)

In [ ]:
from app.agents.nodes.planner import planner_node
for msg in ['Hi', 'How do I configure a Kubernetes ingress?']:
    r = planner_node({'messages': [{'role': 'user', 'content': msg}]})
    print(f'{msg!r:45} -> {r["current_query"]}')

## 4. Portkey (official SDK: `Portkey(api_key, config=...)`)

In [ ]:
from portkey_ai import Portkey
cfg = os.getenv('PORTKEY_PRIMARY_CONFIG_ID')
slug = os.getenv('PORTKEY_PRIMARY_SLUG')
q = [{'role': 'user', 'content': 'In one sentence, how does the Kubernetes Horizontal Pod Autoscaler work?'}]
try:
    if cfg:  # saved config (routing/fallback defined in Portkey UI)
        pk = Portkey(api_key=os.getenv('PORTKEY_API_KEY'), config=cfg)
        r = pk.chat.completions.create(model='@%s/openai/gpt-oss-20b' % slug, messages=q)
    else:    # provider slug only
        pk = Portkey(api_key=os.getenv('PORTKEY_API_KEY'), provider='@' + slug)
        r = pk.chat.completions.create(model='openai/gpt-oss-20b', messages=q)
    print('PASS Portkey ->', r.choices[0].message.content)
except Exception as e:
    print('FAIL Portkey ->', str(e)[:300])

## 5. Portkey via app code (`app.gateway`)

In [ ]:
try:
    from app.gateway import get_langchain_llm
    print('PASS gateway ->', get_langchain_llm('test').invoke('Why is NUMA remote memory access slower than local access? One sentence.').content)
except Exception as e:
    print('FAIL gateway ->', str(e)[:300])

## 6. Gemini embedding (`app.gateway.check_gemini_embedding`)

In [1]:
from app.gateway import check_gemini_embedding
ok, msg = check_gemini_embedding('What is a Kubernetes Pod?')
print('PASS' if ok else 'FAIL', 'Gemini embedding ->', msg)

ModuleNotFoundError: No module named 'app'

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # project root, in case the setup cell wasn't run
from app.gateway import check_gemini_embedding
ok, msg = check_gemini_embedding('What is a Kubernetes Pod?')
print('PASS' if ok else 'FAIL', 'Gemini embedding ->', msg)

PASS Gemini embedding -> dim=768


## 7. Ingestion path: `embed_texts` (used by app/ingestion/processor.py) must use Gemini

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from app.services.retrieval import embeddings as emb
chunks = ['A Kubernetes Pod wraps one or more containers.', 'NUMA gives each CPU socket its own local memory.']
vecs = emb.embed_texts(chunks)
print('model used   :', emb._model_type)
print('vectors      :', len(vecs), 'x', len(vecs[0]))
print('PASS' if emb._model_type == 'gemini' else 'FAIL (fell back to local model)', '- ingestion embeds with Gemini')

c:\dev\Enterprise_RAG\app\services\retrieval\embeddings.py:43: LogfireNotConfiguredWarning: No logs or spans will be created until `logfire.configure()` has been called. Set the environment variable LOGFIRE_IGNORE_NO_CONFIG=1 or add ignore_no_config=true in pyproject.toml to suppress this warning.
  logfire.info(f"✅ Gemini Embedding Model is ready model = gemini-embedding-2, dim = {_GEMINI_DIM}")


model used   : gemini
vectors      : 2 x 768
PASS - ingestion embeds with Gemini


## 8. Retriever path: `embed_query` + Qdrant search (app/services/retrieval/qdrant_service.py)

In [3]:
from app.services.retrieval.embeddings import embed_query
qv = embed_query('How do I scale pods automatically?')
print('model used :', emb._model_type, '| query dim:', len(qv))
print('PASS' if emb._model_type == 'gemini' else 'FAIL (fell back to local model)', '- retriever embeds with Gemini')

# end-to-end: search the Qdrant collection (needs ingested data)
from app.services.retrieval.qdrant_service import search_enterprise_knowledge
try:
    hits = search_enterprise_knowledge('How do I scale pods automatically?', limit=3)
    print('PASS search ->', len(hits), 'hits')
    for h in hits: print(' -', str(h)[:120])
except Exception as e:
    print('FAIL search ->', str(e)[:200])

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-2\nPlease retry in 28.392351567s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 'q